## Executive Summary

**Purpose:** Calculate flux distribution in motor magnetic circuits

**What it does:** Solve magnetic equivalent circuits (MEC) using Newton-Raphson iteration.

### Why It Matters
Quick estimation of motor flux linkage and torque without FEA (replaces lengthy FEA simulations).

### Quick Start
**Inputs:** Circuit topology (branches, nodes), material models, MMF sources, winding definitions

**Outputs:** Flux in each circuit branch, flux linkage per winding, field energy

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Central solver that uses results from 03_* and 04_material_models.ipynb

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple
import math

# Cross-notebook dependencies: import from the installed package so this
# notebook can run independently (nbdev pattern).
from emachines.mec import (
    PermeabilityModel,
    LinearPermeabilityModel,
    SplinePermeabilityModel,
    ShanesudhoffModel,
)

## Theory: Magnetic Equivalent Circuits

A Magnetic Equivalent Circuit (MEC) uses Ohm's law analogy:

- **Reluctance** R [A-turns/Wb] replaces resistance
- **Flux** Φ [Wb] replaces current
- **Magnetomotive force (MMF)** F [A-turns] replaces voltage

Branch equation (linear or nonlinear):
$$F = R(\Phi) \cdot \Phi + F_s$$

where F_s is an optional permanent-magnet (Norton) flux source.

### Reluctance for Nonlinear Materials

For a magnetic core with length ℓ, cross-section A, and permeability μ(B):

$$R = \frac{\ell}{\mu(B) \cdot A}, \quad B = \frac{\Phi}{A}$$

As flux increases, B increases and μ(B) may decrease (saturation), causing R to increase.

### Newton-Raphson Linearization

For each nonlinear branch, the MEC solver builds an effective (incremental) reluctance and source:

$$R_{\text{eff}} = R_0 \cdot (1 - S_0)$$
$$F_{\text{eff}} = F_s - R_0 \cdot (S_0 \cdot \Phi_0 - \Phi_{\text{source}})$$

where
- $R_0 = \ell / (\mu(B_0) \cdot A)$ — operating-point reluctance
- $S_0 = \frac{d\mu/dB \cdot B_0}{\mu}$ — saturation factor
- $B_0 = \Phi_0 / A$ — operating-point flux density

In [2]:
#| export
class BranchType:
    """Branch type identifiers for the MEC."""
    MESH       = "mesh"
    RELUCTANCE = "reluctance"
    NODAL      = "nodal"

In [3]:
#| hide
@dataclass
class _Branch:
    """Internal representation of a single MEC branch."""
    branch_id:    int
    btype:        str
    mesh_id:      Optional[int] = None
    mmf:          float = 0.0
    length:       float = 0.0
    area:         float = 0.0
    model:        Optional[Any] = None
    phi_source:   float = 0.0
    meshes:       List[int] = field(default_factory=list)
    orientations: List[int] = field(default_factory=list)
    permeance:    Optional[float] = None
    node_from:    Optional[int] = None
    node_to:      Optional[int] = None

@dataclass
class _Winding:
    """Internal descriptor of a multi-turn winding."""
    winding_id:   Any
    branch_turns: Dict[int, float] = field(default_factory=dict)

## MEC Solver Implementation

In [ ]:
#| export
MU0 = 4e-7 * math.pi

class MEC:
    """
    Magnetic Equivalent Circuit builder and Newton-Raphson solver.

    Parameters
    ----------
    phi_base : float, optional
        Base flux [Wb] for per-unit scaling (Horvath 2018).
        Typically ``B_base * A_base`` where ``B_base ≈ 1.6 T``.
        If *None*, a heuristic scale is used instead.
    F_base : float, optional
        Base MMF [A-turns] for per-unit scaling.
        Must be supplied together with *phi_base* or not at all.
    rtol, atol : float
        Newton-Raphson convergence tolerances on ‖ΔΦ‖:
        ``‖ΔΦ‖ ≤ rtol·½‖Φₖ + Φₖ₋₁‖ + atol``
    max_iter : int
        Maximum Newton-Raphson iterations.
    """

    def __init__(
        self,
        phi_base: Optional[float] = None,
        F_base: Optional[float] = None,
        rtol: float = 1e-3,
        atol: float = 1e-6,
        max_iter: int = 20,
    ) -> None:
        if (phi_base is None) != (F_base is None):
            raise ValueError("phi_base and F_base must both be specified or both be None.")

        self.phi_base = phi_base
        self.F_base = F_base
        self.rtol = rtol
        self.atol = atol
        self.max_iter = max_iter

        self._branches: Dict[int, _Branch] = {}
        self._windings: Dict[Any, _Winding] = {}
        self._next_branch_id = 0
        self._mesh_count = 0
        self._node_count = 0

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def add_mesh_branch(
        self,
        mmf: float = 0.0,
    ) -> int:
        """Add a mesh (loop current) branch.

        Parameters
        ----------
        mmf : float
            Magnetomotive force source [A-turns].

        Returns
        -------
        branch_id : int
            Unique identifier for this branch.
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.MESH,
            mesh_id=self._mesh_count,
            mmf=mmf,
        )
        self._next_branch_id += 1
        self._mesh_count += 1
        return bid

    def add_reluctance_branch(
        self,
        length: float,
        area: float,
        model: Any,
        meshes: Sequence[int] = (),
        orientations: Sequence[int] = (),
        mmf: float = 0.0,
        phi_source: float = 0.0,
    ) -> int:
        """Add a reluctance (nonlinear magnetic) branch.

        Parameters
        ----------
        length : float
            Core length [m].
        area : float
            Cross-sectional area [m²].
        model : PermeabilityModel
            Permeability model (must have mu() and dmu_dB() methods).
        meshes : sequence of int
            Mesh IDs that cross this branch (±1).
        orientations : sequence of int
            +1 or -1 for each mesh (orientation relative to branch flux).
        mmf : float
            Magnetomotive force source [A-turns].
        phi_source : float
            PM (Norton equivalent) flux source [Wb].

        Returns
        -------
        branch_id : int
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.RELUCTANCE,
            length=length,
            area=area,
            model=model,
            mmf=mmf,
            phi_source=phi_source,
            meshes=list(meshes),
            orientations=list(orientations),
        )
        self._next_branch_id += 1
        return bid

    def add_nodal_branch(
        self,
        permeance: float,
        node_from: int,
        node_to: int,
        mmf: float = 0.0,
    ) -> int:
        """Add a nodal (linear permeance) branch.

        Parameters
        ----------
        permeance : float
            Fixed permeance P = A/μℓ [Wb/A-turns].
        node_from : int
            Starting magnetic node.
        node_to : int
            Ending magnetic node.
        mmf : float
            Magnetomotive force source.

        Returns
        -------
        branch_id : int
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.NODAL,
            node_from=node_from,
            node_to=node_to,
            permeance=permeance,
            mmf=mmf,
        )
        self._node_count = max(self._node_count, node_from + 1, node_to + 1)
        self._next_branch_id += 1
        return bid

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def add_winding(
        self,
        winding_id: Any,
        branch_turns: Dict[int, float],
    ) -> None:
        """Register a winding for flux linkage calculation.

        Parameters
        ----------
        winding_id : Any
            Unique identifier for this winding.
        branch_turns : dict
            Mapping {branch_id: signed_turn_count}.
            Positive = aligned with branch flux direction.
        """
        self._windings[winding_id] = _Winding(
            winding_id=winding_id,
            branch_turns=dict(branch_turns),
        )

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def solve(self) -> Any:  # Returns MECSolution
        """Solve the MEC using Newton-Raphson iteration.

        Returns
        -------
        solution : MECSolution
            Solution object with flux, MMF, field energy, and convergence info.
        """
        # Separate branches by type
        mesh_b      = [b for b in self._branches.values() if b.btype == BranchType.MESH]
        reluctance_b = [b for b in self._branches.values() if b.btype == BranchType.RELUCTANCE]

        # State vector: one entry per mesh variable
        n_mesh = len(mesh_b)
        n_nodal = self._node_count
        n_vars = n_mesh + n_nodal

        # Map branch_id -> column index in Phi
        mesh_idx = {b.branch_id: i for i, b in enumerate(mesh_b)}

        Phi = np.zeros(n_vars)
        converged = False
        residual  = float('inf')
        n_iter    = 0

        for n_iter in range(self.max_iter):
            J     = np.zeros((n_vars, n_vars))
            F_vec = np.zeros(n_vars)
            R_eff_map = {}
            Fs_eff_map = {}

            # Build one KVL equation per mesh
            for i, mesh_branch in enumerate(mesh_b):
                mmf_sum = mesh_branch.mmf  # source MMF for this loop

                for r in reluctance_b:
                    if mesh_branch.branch_id not in r.meshes:
                        continue
                    idx_in_r  = r.meshes.index(mesh_branch.branch_id)
                    orientation = r.orientations[idx_in_r]

                    phi_b = self._branch_flux(r, Phi, mesh_idx)
                    B     = phi_b / r.area if r.area > 0 else 0.0
                    mu    = r.model.mu(B)
                    dmu_dB = r.model.dmu_dB(B)

                    R0    = r.length / (mu * r.area) if mu > 0 and r.area > 0 else float('inf')
                    S0    = (dmu_dB * B / mu) if mu > 0 else 0.0
                    R_eff = R0 * (1.0 - S0) if R0 < float('inf') else float('inf')
                    Fs_eff = r.mmf - R0 * (S0 * phi_b - r.phi_source)

                    R_eff_map[r.branch_id]  = R_eff
                    Fs_eff_map[r.branch_id] = Fs_eff

                    # KVL residual: f_i = MMF_source - (R_eff*phi - Fs_eff)
                    mmf_sum -= R_eff * phi_b - Fs_eff

                    # Jacobian: df_i / d(Phi_mesh_i)
                    if R_eff < float('inf'):
                        J[i, i] -= orientation * R_eff

                F_vec[i] = mmf_sum

            # Solve J * dPhi = F  (Newton step)
            try:
                dPhi = np.linalg.solve(J + np.eye(n_vars) * 1e-12, F_vec)
            except np.linalg.LinAlgError:
                dPhi = np.linalg.pinv(J) @ F_vec

            Phi_old = Phi.copy()
            Phi    -= dPhi

            norm_Phi = 0.5 * (np.linalg.norm(Phi) + np.linalg.norm(Phi_old))
            residual = np.linalg.norm(dPhi)
            if residual <= self.rtol * norm_Phi + self.atol:
                converged = True
                break

        # ── Post-processing ──────────────────────────────────────────────────
        phi_mesh     = {b.branch_id: Phi[mesh_idx[b.branch_id]] for b in mesh_b}
        phi_branches = {}
        mmf_branches = {}

        for b in mesh_b:
            phi_branches[b.branch_id] = phi_mesh[b.branch_id]
            mmf_branches[b.branch_id] = b.mmf

        for b in reluctance_b:
            phi_b = self._branch_flux(b, Phi, mesh_idx)
            phi_branches[b.branch_id] = phi_b
            R_eff  = R_eff_map.get(b.branch_id, 0.0)
            Fs_eff = Fs_eff_map.get(b.branch_id, b.mmf)
            mmf_branches[b.branch_id] = R_eff * phi_b - Fs_eff

        flux_linkages = {
            wid: sum(N * phi_branches.get(bid, 0.0)
                     for bid, N in w.branch_turns.items())
            for wid, w in self._windings.items()
        }

        Wf = sum(
            0.5 * R_eff_map.get(b.branch_id, 0.0) * phi_branches[b.branch_id] ** 2
            for b in reluctance_b
        )

        try:
            from .result import MECSolution         # generated module context
        except ImportError:
            from emachines.mec import MECSolution   # interactive notebook context
        return MECSolution(
            phi_mesh=phi_mesh,
            phi_nodal={},
            phi_branches=phi_branches,
            mmf_branches=mmf_branches,
            flux_linkages=flux_linkages,
            converged=converged,
            n_iterations=n_iter + 1,
            residual=residual,
            field_energy=Wf,
            phi_base=self.phi_base,
            F_base=self.F_base,
        )

    def _branch_flux(self, branch, Phi, mesh_idx) -> float:
        """Return net flux through *branch* from current mesh variables."""
        return sum(
            o * Phi[mesh_idx[m]]
            for m, o in zip(branch.meshes, branch.orientations)
            if m in mesh_idx
        )

## Example: Simple Series Reluctance Circuit

In [8]:
# Import the material model
# LinearPermeabilityModel is imported in the setup cell above.
# (from emachines.mec import LinearPermeabilityModel)

# Create a simple MEC with a single mesh and reluctance
mec = MEC()

# Add a mesh with 100 A-turns source
mesh1 = mec.add_mesh_branch(mmf=100.0)

# Add an air-gap reluctance (constant permeability)
air_model = LinearPermeabilityModel(mu_r=1.0)  # Free space
reluctance1 = mec.add_reluctance_branch(
    length=0.001,  # 1 mm gap
    area=0.01,     # 100 mm² area
    model=air_model,
    meshes=[mesh1],
    orientations=[1],
)

# Solve
solution = mec.solve()
print(f"Air-gap flux: {solution.flux(reluctance1):.6e} Wb")
print(f"Converged: {solution.converged}")
print(f"Iterations: {solution.n_iterations}")

Air-gap flux: 1.256637e-03 Wb
Converged: True
Iterations: 2


## Tests

In [9]:
# Test MEC creation
mec = MEC()
assert mec.rtol == 1e-3, "Default rtol should be 1e-3"
assert mec.atol == 1e-6, "Default atol should be 1e-6"
print("✓ MEC initialization tests passed")

# Test branch addition
mesh_id = mec.add_mesh_branch(mmf=50.0)
assert mesh_id == 0, "First branch should have ID 0"

air_model = LinearPermeabilityModel(mu_r=1.0)
reluc_id = mec.add_reluctance_branch(
    length=0.001,
    area=0.01,
    model=air_model,
    meshes=[mesh_id],
    orientations=[1],
)
assert reluc_id == 1, "Second branch should have ID 1"
print("✓ Branch addition tests passed")

# Test winding registration
mec.add_winding('coil_a', {reluc_id: 10})  # 10 turns linked with reluctance
assert 'coil_a' in mec._windings, "Winding should be registered"
print("✓ Winding registration tests passed")

# Test simple solve
try:
    sol = mec.solve()
    assert hasattr(sol, 'converged'), "Solution should have 'converged' attribute"
    assert hasattr(sol, 'n_iterations'), "Solution should have 'n_iterations' attribute"
    print("✓ Solve method tests passed")
except Exception as e:
    print(f"Note: Solve test encountered {type(e).__name__}: {e}")

✓ MEC initialization tests passed
✓ Branch addition tests passed
✓ Winding registration tests passed
✓ Solve method tests passed
